In [2]:
!pip install anndata==0.8.0

  Using cached anndata-0.8.0-py3-none-any.whl (96 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 28.4 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: h5py
    Found existing installation: h5py 2.10.0
    Uninstalling h5py-2.10.0:
      Successfully uninstalled h5py-2.10.0
  Attempting uninstall: anndata
    Found existing installation: anndata 0.7.8
    Uninstalling anndata-0.7.8:
      Successfully uninstalled anndata-0.7.8
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
samap 1.0.2 requires h5py<=2.10, but you have h5py 3.8.0 which is incompatible.
sam-algorithm 1.0.0 requires h5py<=2.10.0, but you have h5py 3.8.0 which is incompatible.


In [2]:
from samap.mapping import SAMAP
from samap.analysis import (get_mapping_scores, GenePairFinder, transfer_annotations,
                            sankey_plot, chord_plot, CellTypeTriangles, 
                            ParalogSubstitutions, FunctionalEnrichment,
                            convert_eggnog_to_homologs, GeneTriangles)
from samalg import SAM
import pandas as pd
from Bio import SeqIO
from samap.utils import (save_samap, load_samap)
import scanpy as sc
import matplotlib.colors
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from scipy import sparse 
from scipy import cluster
import seaborn as sns
import random
import sklearn
from scipy.stats import poisson
from sklearn.neighbors import KernelDensity
import time
import dill
from scipy.optimize import minimize
import pickle
import itertools
import os
import anndata as ad

In [3]:
with open('parent_dict_v2.pkl', 'rb') as f:
    parent_dict = pickle.load(f)

In [4]:
fn = 'Active_SAM_joined/SAM_MO_soupx_plus5_cleaned_NN_04142026.h5ad'
sam_mo = SAM()
sam_mo.load_data(fn)
gene_dict_mo = {}
for i in range(len(sam_mo.adata.var_names)):
    gene_dict_mo[sam_mo.adata.var_names[i]] = i
gene_dict_mo['NaN'] = 'NaN'

In [5]:
fn = 'Active_SAM_joined/SAM_CJ_joined_v2_cleaned_03122025.h5ad'
sam_cj = SAM()
sam_cj.load_data(fn)
gene_dict_cj = {}
for i in range(len(sam_cj.adata.var_names)):
    gene_dict_cj[sam_cj.adata.var_names[i]] = i
gene_dict_cj['NaN'] = 'NaN'

In [6]:
fn = 'Active_SAM_joined/SAM_AC_ncbi_soupx_cleaned_03122025.h5ad'
sam_ac = SAM()
sam_ac.load_data(fn)
gene_dict_ac = {}
for i in range(len(sam_ac.adata.var_names)):
    gene_dict_ac[sam_ac.adata.var_names[i]] = i
gene_dict_ac['NaN'] = 'NaN'

In [7]:
fn = 'Active_SAM_joined/SAM_XT_joined_Slc17a6_cleaned_03122205.h5ad'
sam_xt = SAM()
sam_xt.load_data(fn)
gene_dict_xt = {}
for i in range(len(sam_xt.adata.var_names)):
    gene_dict_xt[sam_xt.adata.var_names[i]] = i
gene_dict_xt['NaN'] = 'NaN'

In [8]:
fn = 'Active_SAM_joined/SAM_DR_ncbi_joined_cleaned_07172026.h5ad'
sam_dr = SAM()
sam_dr.load_data(fn)
gene_dict_dr = {}
for i in range(len(sam_dr.adata.var_names)):
    gene_dict_dr[sam_dr.adata.var_names[i]] = i
gene_dict_dr['NaN'] = 'NaN'

In [10]:
far_classes = [1, 2, 3, 4, 5, 6, 7, 17, 19, 20, 21, 24, 25, 27, 28,29]

In [11]:
sam_cj.adata.obs.columns

Index(['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'n_genes', 'n_counts',
       'key', 'hicat_merged', 'subclass_id_label_mapping',
       'subclass_id_label_lc', 'leiden_clusters',
       'subclass_id_label_mapping_nounlabeled', 'neurotransmitter',
       'region_label', 'subclass_id_label_reduced_mapping',
       'subclass_id_label_reduced_lc',
       'subclass_id_label_reduced_mapping_nounlabeled', 'fraction_match',
       'best_match', 'frac_match_test', 'leiden_removal', 'nCount_SCT',
       'nFeature_SCT', 'SCT_snn_res.0.8', 'seurat_clusters', 'SCT_snn_res.5',
       'eq_subclass', 'eq_subclass_lc', 'eq_subclass_frac',
       'eq_subclass_nounlabeled', 'eq_subclass_nounlabeled_NN',
       'eq_subclass_nounlabeled_nmm', 'ss_subclass', 'ss_subclass_nounlabeled',
       'ss_class', 'ss_subclass_nounlabeled_astro', 'ss_subclass_v2',
       'ss_subclass_v2_nounlabeled', 'ss_subclass_nounlabeled_nmm',
       'ss_subclass_v3_nounlabeled', 'subclass_id_label_crossed',
       'ss_subclas

In [13]:
#Number of cells
vole_far = 0
vole_close = 0
vole_hypo = 0
vole_close_name = []
for item in sam_mo.adata.obs['ss_subclass_nounlabeled_03102026'].unique():
    if item in parent_dict and 'NN' not in item and item != 'Unlabeled':
        if int(parent_dict[item][:2]) in far_classes:
            vole_far += len(sam_mo.adata[sam_mo.adata.obs['ss_subclass_nounlabeled_03102026'] == item])
        elif parent_dict[parent_dict[item]] != 'hypo':
            vole_close_name.append(item)
            vole_close += len(sam_mo.adata[sam_mo.adata.obs['ss_subclass_nounlabeled_03102026'] == item])
        else:
            vole_hypo += len(sam_mo.adata[sam_mo.adata.obs['ss_subclass_nounlabeled_03102026'] == item])
cj_far = 0
cj_close = 0
cj_hypo = 0
cj_close_name = []
for item in sam_cj.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn'].unique():
    if item in parent_dict and 'NN' not in item and item != 'Unlabeled':
        if int(parent_dict[item][:2]) in far_classes:
            cj_far += len(sam_cj.adata[sam_cj.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn'] == item])
        elif parent_dict[parent_dict[item]] != 'hypo':
            cj_close_name.append(item)
            cj_close += len(sam_cj.adata[sam_cj.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn'] == item])
        else:
            cj_hypo += len(sam_cj.adata[sam_cj.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn'] == item])
            
ac_far = 0
ac_close = 0
ac_hypo = 0 
ac_close_name = []
for item in sam_ac.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn'].unique():
    if item in parent_dict and 'NN' not in item and item != 'Unlabeled':
        if int(parent_dict[item][:2]) in far_classes:
            ac_far += len(sam_ac.adata[sam_ac.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn'] == item])
        elif parent_dict[parent_dict[item]] != 'hypo':
            ac_close_name.append(item)
            ac_close += len(sam_ac.adata[sam_ac.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn'] == item])
        else:
            ac_hypo += len(sam_ac.adata[sam_ac.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn'] == item])
            

xt_far = 0
xt_close = 0
xt_hypo = 0
xt_close_name = []
for item in sam_xt.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn'].unique():
    if item in parent_dict and 'NN' not in item and item != 'Unlabeled':
        if int(parent_dict[item][:2]) in far_classes:
            xt_far += len(sam_xt.adata[sam_xt.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn'] == item])
        elif parent_dict[parent_dict[item]] != 'hypo':
            xt_close_name.append(item)
            xt_close += len(sam_xt.adata[sam_xt.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn'] == item])
        else:
            xt_hypo += len(sam_xt.adata[sam_xt.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn'] == item])
            
            
dr_far = 0
dr_close = 0
dr_hypo = 0
dr_close_name = []
for item in sam_dr.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn'].unique():
    if item in parent_dict and 'NN' not in item and item != 'Unlabeled':
        if int(parent_dict[item][:2]) in far_classes:
            dr_far += len(sam_dr.adata[sam_dr.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn'] == item])
        elif parent_dict[parent_dict[item]] != 'hypo':
            dr_close_name.append(item)
            dr_close += len(sam_dr.adata[sam_dr.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn'] == item])
        else:
            dr_hypo += len(sam_dr.adata[sam_dr.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn'] == item])

In [14]:
print(vole_close/len(sam_mo.adata.obs))
print(cj_close/len(sam_cj.adata.obs))
print(ac_close/len(sam_ac.adata.obs))
print(xt_close/len(sam_xt.adata.obs))
print(dr_close/len(sam_dr.adata.obs))

0.09938476100331282
0.09845100189946226
0.1351983298538622
0.06371237062125483
0.01320201855770796


In [15]:
print(vole_far/len(sam_mo.adata.obs))
print(cj_far/len(sam_cj.adata.obs))
print(ac_far/len(sam_ac.adata.obs))
print(xt_far/len(sam_xt.adata.obs))
print(dr_far/len(sam_dr.adata.obs))

0.04347238185383003
0.09462532437999946
0.13198329853862212
0.062101797683617156
0.03812469477454013


In [16]:
dr_close_name

['060 OT D3 Folh1 Gaba',
 '056 Sst Chodl Gaba',
 '054 STR Prox1 Lhx6 Gaba',
 '062 STR D2 Gaba']

In [17]:
sc_ac = ['204 SC Otx2 Gcnt4 Gaba',
 '205 SC-PAG Lef1 Emx2 Gaba',
 '208 SC Lef1 Otx2 Gaba',
 '209 SCs Pax7 Nfia Gaba',
 '211 SC Tnnt1 Gli3 Gaba',
 '212 SCs Lef1 Gli3 Gaba',
 '213 SCsg Gabrr2 Gaba',
'175 SC Bnc2 Glut',
 '186 SCop Pou4f2 Neurod2 Glut',
 '187 SCsg Pde5a Glut',
 '188 SCop Sln Glut',]

In [19]:
len(sam_ac.adata[sam_ac.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn'].isin(sc_ac)])/len(sam_ac.adata)

0.05945720250521921

In [20]:
sc_cj = ['205 SC-PAG Lef1 Emx2 Gaba',
 '207 SCs Dmbx1 Gaba',
 '208 SC Lef1 Otx2 Gaba',
 '209 SCs Pax7 Nfia Gaba',
 '211 SC Tnnt1 Gli3 Gaba',
 '212 SCs Lef1 Gli3 Gaba',
 '213 SCsg Gabrr2 Gaba']

In [21]:
len(sam_cj.adata[sam_cj.adata.obs['ss_subclass_nounlabeled_nmm_cl_v4_nn'].isin(sc_cj)])/len(sam_cj.adata)

0.02213809893255571